# Stage 0 walkthrough: talking to SEC EDGAR politely

Stage 0 is one HTTP client and the rules it enforces *itself*, so no later caller
has to remember them: **identity** (a real User-Agent), **pacing** (≤5 req/s),
and **caching** (every response on disk, forever). Plus the CIK trap that fails far from its cause.

Every cell calls the project's own code. With a warm cache this notebook makes
**zero requests to SEC** — the retry demo in cell 7 uses a fake server.

**Prerequisites:** `uv sync --group notebook`, the `.venv` kernel, and a `.env`
with your `EDGAR_USER_AGENT`. Companion reading: `docs/stage0_foundations.md`.

## 0 · Setup

`Settings` is loaded but never printed — it holds your contact address, and this repo is public.

In [ ]:
import os
from pathlib import Path

# .env, config/ and data/ are relative to the repo root, exactly as when running `fc`.
if Path.cwd().name == "notebooks":
    os.chdir("..")

import hashlib
import json
import time

import pandas as pd

from filing_copilot.config import get_settings

settings = get_settings()
print("settings loaded; rate limit", settings.edgar_rate_limit, "req/s, max retries", settings.edgar_max_retries)

## 1 · Identity is checked at startup, not at request time

SEC requires a User-Agent with a real contact. `Settings` refuses to exist without
one — the placeholder from `.env.example`, an empty string, or text with no email
all fail *before* any request could be made under a fake identity.

In [ ]:
from pydantic import ValidationError

from filing_copilot.config import PLACEHOLDER_USER_AGENT, Settings

for candidate in [PLACEHOLDER_USER_AGENT, "", "Just A Name, no contact"]:
    try:
        Settings(edgar_user_agent=candidate)
        print(f"{candidate!r:45} accepted")
    except ValidationError as exc:
        print(f"{candidate!r:45} REFUSED: {exc.errors()[0]['msg']}")

## 2 · One company, three CIKs

SEC renders the same identifier three ways. `company_tickers.json` even calls it
`cik_str` while storing an **int**. Everything inside this project uses one canonical
form — 10 digits, zero-padded, no prefix — converted at the boundary and nowhere else.

In [ ]:
from filing_copilot.edgar import (
    EdgarClient, InvalidCIKError, TickerResolver, normalize_cik, to_archives_cik, to_data_api_cik,
)
from filing_copilot.edgar.endpoints import company_tickers_url

client = EdgarClient(settings)
raw_tickers = client.get(company_tickers_url())          # served from the cache

row = next(r for r in json.loads(raw_tickers).values() if r["ticker"] == "SYF")
print("raw row from SEC:", row, "   type(cik_str) =", type(row["cik_str"]).__name__)

resolver = TickerResolver.from_json(raw_tickers)
print(f"{len(resolver):,} tickers;", resolver.by_ticker("syf"))
print()
for rendering in [1601712, "1601712", "0001601712", "CIK0001601712"]:
    print(f"normalize_cik({rendering!r:17}) -> {normalize_cik(rendering)!r}")
print()
print("data.sec.gov form:", to_data_api_cik("1601712"), "   Archives form:", to_archives_cik("0001601712"))
try:
    normalize_cik("16017120000001")
except InvalidCIKError as exc:
    print("rejected:", exc)

## 3 · Every URL is spelled exactly once

`www.sec.gov` and `data.sec.gov` are different hosts with different CIK conventions.
`edgar/endpoints.py` builds every URL, so nobody hand-assembles one and drops the padding.

In [ ]:
from filing_copilot.edgar.endpoints import (
    companyfacts_bulk_url, companyfacts_url, filing_document_url, submissions_url,
)

urls = {
    "ticker map":        company_tickers_url(),
    "submissions":       submissions_url("1601712"),
    "companyfacts (1)":  companyfacts_url("1601712"),
    "companyfacts bulk": companyfacts_bulk_url(),
    "a 10-K document":   filing_document_url("1601712", "0001601712-26-000006", "syf-20251231.htm"),
}
for name, url in urls.items():
    print(f"{name:18} {url}")

## 4 · The cache layout mirrors the URL

Paths are derived from the URL, not hashed (ADR-0013), so you can find any response
by reading its address. Each body has a `.meta.json` sidecar with a SHA-256 — the link
from a future citation back to the exact bytes SEC served. Recomputed here to prove it.

In [ ]:
cache = client.cache
for name, url in urls.items():
    path = cache.path_for(url)
    print(f"{name:18} {path}   {'cached' if path.is_file() else '-'}")

url = urls["submissions"]
meta = cache.meta_for(url)
print()
print(json.dumps(meta, indent=2))
recomputed = hashlib.sha256(cache.path_for(url).read_bytes()).hexdigest()
print("\nsha256 on disk matches sidecar:", recomputed == meta["sha256"])

## 5 · A warm cache makes zero requests

`request_count` counts real network requests. Every stage asserts it is zero on a
warm cache — that is how "cache everything" is enforced rather than hoped for.

In [ ]:
before = client.request_count
started = time.perf_counter()
for _ in range(3):
    client.get(urls["submissions"])
print(f"3 gets in {(time.perf_counter() - started) * 1000:.1f} ms, network requests: {client.request_count - before}")

## 6 · The rate limiter — ≤5 requests per second

A monotonic clock (immune to system clock changes) and a lock (safe once downloads
run in parallel). Eleven acquisitions at 5/s must take about 2 seconds: the first is free,
each of the other ten waits 0.2s.

In [ ]:
from filing_copilot.edgar import RateLimiter

limiter = RateLimiter(settings.edgar_rate_limit)
stamps = []
for _ in range(11):
    limiter.acquire()
    stamps.append(time.monotonic())

gaps = [b - a for a, b in zip(stamps, stamps[1:])]
print(f"11 acquisitions in {stamps[-1] - stamps[0]:.2f}s; gaps {min(gaps):.3f}-{max(gaps):.3f}s "
      f"(min interval {limiter.min_interval:.3f}s)")

## 7 · Retries — against a fake SEC, so nothing real is hit

`httpx.MockTransport` stands in for SEC, exactly as the test suite does. The fake
returns 503, then 503 with `Retry-After: 3`, then 200. Watch three things:

- **every retry is a request** and passes through the rate limiter (`limiter waits`) —
  wrapping the throttle *outside* a retry loop would let retries burst past 5/s;
- backoff honours `Retry-After` when sent, else exponential with jitter;
- a **404 is never retried** — it will still be a 404 next time.

Sleeps are recorded rather than slept, and the cache is a temporary directory.

In [ ]:
import tempfile

import httpx

from filing_copilot.edgar import EdgarHTTPError, ResponseCache

script = [
    httpx.Response(503),
    httpx.Response(503, headers={"Retry-After": "3"}),
    httpx.Response(200, json={"hello": "from a fake SEC"}),
]
seen = []

def fake_sec(request: httpx.Request) -> httpx.Response:
    seen.append((request.method, request.url.path, request.headers["user-agent"] != ""))
    return script.pop(0) if request.url.path.endswith("0000000001.json") else httpx.Response(404)

backoff_sleeps, limiter_waits = [], []
with tempfile.TemporaryDirectory() as tmp:
    fake = EdgarClient(
        settings,
        cache=ResponseCache(Path(tmp)),
        limiter=RateLimiter(settings.edgar_rate_limit, sleep=limiter_waits.append),
        transport=httpx.MockTransport(fake_sec),
        sleep=backoff_sleeps.append,
    )
    body = fake.get(submissions_url("1"))
    print("body:", body, "  requests:", fake.request_count)
    print("backoff sleeps:", [round(s, 2) for s in backoff_sleeps], " <- 2nd honours Retry-After: 3")
    print("limiter waits: ", [round(s, 3) for s in limiter_waits], " <- retries consumed rate budget")
    print("every request carried a User-Agent:", all(ua for _, _, ua in seen))

    fake.request_count = 0
    try:
        fake.get(submissions_url("2"))
    except EdgarHTTPError as exc:
        print("\n404 ->", exc, "  requests:", fake.request_count)

## 8 · What SEC gives you: `submissions.json`

A company's metadata plus its filing history. Note `filings.recent` is **columnar**
(a dict of parallel lists) and **truncated** at ~1,000 rows — older filings live in the
pages listed under `filings.files`. Synchrony's window is mostly Form 4s; that is why
Stage 2 has to page back through history.

In [ ]:
submissions = json.loads(client.get(urls["submissions"]))
print("top-level keys:", list(submissions))
print("fiscal year end:", submissions["fiscalYearEnd"], "  SIC:", submissions["sicDescription"])

recent = pd.DataFrame(submissions["filings"]["recent"])
print(f"\nfilings.recent: {len(recent):,} rows, {recent['filingDate'].min()} to {recent['filingDate'].max()}")
print(recent["form"].value_counts().head(8).to_dict())
print("\nolder pages:", submissions["filings"]["files"])
recent[recent["form"] == "10-K"][["accessionNumber", "filingDate", "reportDate", "primaryDocument"]]

## 9 · The bulk archive — read in place, never extracted

One request for every company's XBRL facts instead of one per company (ADR-0016).
It is ~1.3 GB compressed and **~18 GiB extracted** (measured below), which will not fit
on this disk — so Stage 1 reads the 20 members it needs directly out of the zip.

In [ ]:
import zipfile

from filing_copilot.edgar.endpoints import companyfacts_bulk_url

archive_path = cache.path_for(companyfacts_bulk_url())
with zipfile.ZipFile(archive_path) as archive:
    members = archive.infolist()
    syf = archive.getinfo("CIK0001601712.json")

print(f"{archive_path}: {archive_path.stat().st_size / 2**30:.2f} GiB on disk")
print(f"{len(members):,} members, {sum(m.file_size for m in members) / 2**30:.1f} GiB if extracted")
print(f"Synchrony's member: {syf.compress_size / 2**20:.1f} MiB compressed, {syf.file_size / 2**20:.1f} MiB raw")

## What Stage 0 outputs

| Output | Where | Used by |
|---|---|---|
| A polite `EdgarClient` — identity, pacing, retries, caching | `edgar/client.py` | every later stage |
| The response cache, mirroring SEC's URLs, with sha256 sidecars | `data/raw/` | Stage 1 (bulk zip), Stage 2 (submissions, filings) |
| One canonical CIK form and one place for URLs | `edgar/identifiers.py`, `edgar/endpoints.py` | everything that names a company |

In [ ]:
stats = cache.stats()
print(f"cache: {stats.entries:,} entries, {stats.total_bytes / 2**20:,.1f} MiB, {stats.oldest:%Y-%m-%d} to {stats.newest:%Y-%m-%d}")
print("network requests this whole notebook:", client.request_count)
client.close()